## Binary Customer Churn

A marketing agency has many customers that use their service to produce ads for the client/customer websites. They've noticed that they have quite a bit of churn in clients. They basically randomly assign account managers right now, but want you to create a machine learning model that will help predict which customers will churn (stop buying their service) so that they can correctly assign the customers most at risk to churn an account manager. Luckily they have some historical data, can you help them out? Create a classification algorithm that will help classify whether or not a customer churned. Then the company can test this against incoming data for future customers to predict which customers will churn and assign them an account manager.

The data is saved as customer_churn.csv. Here are the fields and their definitions:

    Name : Name of the latest contact at Company
    Age: Customer Age
    Total_Purchase: Total Ads Purchased
    Account_Manager: Binary 0=No manager, 1= Account manager assigned
    Years: Totaly Years as a customer
    Num_sites: Number of websites that use the service.
    Onboard_date: Date that the name of the latest contact was onboarded
    Location: Client HQ Address
    Company: Name of Client Company
    
Once you've created the model and evaluated it, test out the model on some new data (you can think of this almost like a hold-out set) that your client has provided, saved under new_customers.csv. The client wants to know which customers are most likely to churn given this data (they don't have the label yet).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
df = pd.read_csv('customer_churn.csv')
df.head(2)

In [ ]:
# drop column Names, Location and Company
# removing the columns that are not needed for the model building
# only numerical data is needed for the model building

df.drop(['Names','Location','Company', 'Onboard_date'],axis=1,inplace=True)

In [ ]:
#Feature Engineering: Create Average Annual Purchase
# Combining Total_Purchase and Years to create a new, powerful signal.
df['Avg_Annual_Purchase'] = df['Total_Purchase'] / df['Years']
df.head()

In [ ]:
#Outlier Handling (IQR Method) for Total_Purchase
Q1 = df['Total_Purchase'].quantile(0.25)
Q3 = df['Total_Purchase'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Cap outliers to the upper and lower bounds
df['Total_Purchase'] = np.where(df['Total_Purchase'] > upper_bound, upper_bound, df['Total_Purchase'])
df['Total_Purchase'] = np.where(df['Total_Purchase'] < lower_bound, lower_bound, df['Total_Purchase'])

In [ ]:
#Correlation Checking & Multicollinearity
# Check for highly correlated features to drop if necessary.
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# datatypes of columns
df.dtypes

In [ ]:
# sigmoid function
def sigmoid(z):
    return 1/(1+np.exp(-z))


In [ ]:
# cost function
def costFunction(X, y, theta):
    m = len(y)
    h = sigmoid(X.dot(theta))
    error = (y*np.log(h) + (1-y)*np.log(1-h))
    cost = -1/m * sum(error)
    grad = 1/m * X.T.dot(h-y)
    return cost, grad


In [ ]:
# gradient descent
def gradientDescent(X, y, theta, alpha, iterations):
    cost_history = np.zeros(iterations)
    for i in range(iterations):
        cost, grad = costFunction(X, y, theta)
        theta -= alpha * grad
        cost_history[i] = cost
    return theta, cost_history


In [ ]:
# predict function
def predict(X, theta):
    return sigmoid(X.dot(theta))

In [ ]:
# accuracy function
def accuracy(y_true, y_pred):
    accuracy = np.sum(y_true == y_pred) / len(y_true)
    return accuracy

In [ ]:
# train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('Churn', axis=1), df['Churn'], test_size=0.2, random_state=42)

In [ ]:
# feature scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
# add intercept
X_train = np.c_[np.ones((X_train.shape[0], 1)), X_train]
X_test = np.c_[np.ones((X_test.shape[0], 1)), X_test]


In [ ]:
# train model
theta, cost_history = gradientDescent(X_train, y_train, np.zeros(X_train.shape[1]), alpha=0.01, iterations=1000)


In [ ]:
# plot cost history
plt.plot(range(1000), cost_history)
plt.xlabel('Iterations')
plt.ylabel('Cost')
plt.title('Cost history')


In [ ]:
# predict
y_pred = predict(X_test, theta)
y_pred = [1 if i > 0.5 else 0 for i in y_pred]


In [ ]:
# accuracy
accuracy(y_test, y_pred)*100

In [ ]:
df2 = pd.read_csv('new_customers_1.csv')
df2.head(2)

In [ ]:
df2.drop(['Names','Location','Company', 'Onboard_date'],axis=1,inplace=True)

In [ ]:
df2

In [ ]:
# apply prediction on new data
# First, apply the same feature engineering to the new data
df2['Avg_Annual_Purchase'] = df2['Total_Purchase'] / df2['Years']

X_new = scaler.transform(df2)
X_new = np.c_[np.ones((X_new.shape[0], 1)), X_new]
y_pred_new = predict(X_new, theta)
y_pred_new = [1 if i > 0.5 else 0 for i in y_pred_new]
y_pred_new

# insert prediction into dataframe
df2['Churn'] = y_pred_new
df2